<a href="https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/dev/models/deep_learning/DLinear/model_experiment_DLinear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLinear hyperparameter tuning — best architecture v1

Best manual DLinear architecture so far:

```text
DLinear + Store-Dept series_bias
```

Best manual result:
- v1 WMAE: `1506.28`
- v5 WMAE: `1507.44`
- v6 external covariates WMAE: `1548.03`

This notebook does Optuna tuning on the v1 architecture only. It does not add calendar, Store/Dept embeddings, or external covariates.

In [ ]:
%pip install -q "torch>=2.3,<3" "wandb>=0.19,<1" "optuna>=4,<5" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4"

In [ ]:
from __future__ import annotations

import json
import math
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import wandb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_columns", 100)
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})

In [ ]:
CONFIG = {
    "seed": SEED,
    "validation_weeks": 39,
    "holiday_weight": 5.0,
    "max_epochs": 80,
    "patience": 12,
    "scheduler_patience": 4,
    "num_workers": 2,
    "clip_grad_norm": 1.0,
    "n_trials": 20,
    "run_final_best": True,
    "baseline_dlinear_39w_wmae": 1523.209716796875,
    "best_manual_v1_wmae": 1506.282470703125,
    "seasonal_naive_wmae_reference": 1604.2697073319134,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "wandb_entity": "kende23-n-a",
    "wandb_group": "dlinear-hyperparameter-tuning",
}

DATA_DIR = Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR = Path("/content/artifacts/dlinear_hyperparameter_tuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")

## Load data and build target panel

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
missing_train = required_train.difference(train_raw.columns)
missing_test = required_test.difference(test_raw.columns)
if missing_train or missing_test:
    raise ValueError({"missing_train": sorted(missing_train), "missing_test": sorted(missing_test)})

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

all_train_dates = pd.Index(sorted(train_raw["Date"].unique()), name="Date")
test_dates = pd.Index(sorted(test_raw["Date"].unique()), name="Date")
val_dates = all_train_dates[-CONFIG["validation_weeks"]:]
fit_dates = all_train_dates[:-CONFIG["validation_weeks"]]
split_pos = len(fit_dates)

sales_panel = (
    train_raw.pivot_table(index=["Store", "Dept"], columns="Date", values="Weekly_Sales", aggfunc="sum")
    .reindex(columns=all_train_dates)
    .fillna(0.0)
    .sort_index()
)

holiday_by_date = (
    train_raw[["Date", "IsHoliday"]]
    .drop_duplicates("Date")
    .set_index("Date")
    .reindex(all_train_dates)["IsHoliday"]
    .fillna(False)
    .astype(bool)
)

values = sales_panel.to_numpy(dtype=np.float32)
holiday_flags = holiday_by_date.to_numpy(dtype=bool)

print({
    "n_series": len(sales_panel),
    "n_dates": len(all_train_dates),
    "fit_range": (str(fit_dates.min().date()), str(fit_dates.max().date())),
    "validation_range": (str(val_dates.min().date()), str(val_dates.max().date())),
    "test_horizon": len(test_dates),
})

## Metric, datasets, and model

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray, holiday_weight: float = 5.0) -> float:
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(weights))


class WindowDataset(Dataset):
    def __init__(self, values: np.ndarray, holiday_flags: np.ndarray, input_len: int, pred_len: int, end_pos: int):
        self.values = values.astype(np.float32)
        self.holiday_flags = holiday_flags.astype(bool)
        self.input_len = int(input_len)
        self.pred_len = int(pred_len)
        self.end_pos = int(end_pos)
        self.index = []
        max_start = self.end_pos - self.input_len - self.pred_len
        if max_start < 0:
            raise ValueError("Not enough history for configured windows.")
        for series_idx in range(self.values.shape[0]):
            for start in range(max_start + 1):
                self.index.append((series_idx, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx: int):
        series_idx, start = self.index[idx]
        x = self.values[series_idx, start:start + self.input_len]
        y = self.values[series_idx, start + self.input_len:start + self.input_len + self.pred_len]
        mean = x.mean(dtype=np.float64).astype(np.float32)
        std = x.std(dtype=np.float64).astype(np.float32)
        std = np.float32(max(float(std), 1.0))
        target_positions = np.arange(start + self.input_len, start + self.input_len + self.pred_len)
        weights = np.where(self.holiday_flags[target_positions], CONFIG["holiday_weight"], 1.0).astype(np.float32)
        return {
            "x": torch.from_numpy(((x - mean) / std)[:, None]),
            "y": torch.from_numpy((y - mean) / std),
            "weights": torch.from_numpy(weights),
            "mean": torch.tensor(mean, dtype=torch.float32),
            "std": torch.tensor(std, dtype=torch.float32),
            "series_idx": torch.tensor(series_idx, dtype=torch.long),
        }


class ValidationDataset(Dataset):
    def __init__(self, values: np.ndarray, holiday_flags: np.ndarray, input_len: int, pred_len: int, split_pos: int):
        self.values = values.astype(np.float32)
        self.holiday_flags = holiday_flags.astype(bool)
        self.input_len = int(input_len)
        self.pred_len = int(pred_len)
        self.split_pos = int(split_pos)

    def __len__(self):
        return self.values.shape[0]

    def __getitem__(self, series_idx: int):
        start = self.split_pos - self.input_len
        x = self.values[series_idx, start:self.split_pos]
        y = self.values[series_idx, self.split_pos:self.split_pos + self.pred_len]
        mean = x.mean(dtype=np.float64).astype(np.float32)
        std = x.std(dtype=np.float64).astype(np.float32)
        std = np.float32(max(float(std), 1.0))
        target_positions = np.arange(self.split_pos, self.split_pos + self.pred_len)
        weights = np.where(self.holiday_flags[target_positions], CONFIG["holiday_weight"], 1.0).astype(np.float32)
        return {
            "x": torch.from_numpy(((x - mean) / std)[:, None]),
            "y": torch.from_numpy((y - mean) / std),
            "weights": torch.from_numpy(weights),
            "mean": torch.tensor(mean, dtype=torch.float32),
            "std": torch.tensor(std, dtype=torch.float32),
            "series_idx": torch.tensor(series_idx, dtype=torch.long),
        }


class MovingAverage(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.avg = nn.AvgPool1d(kernel_size=self.kernel_size, stride=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad_left = (self.kernel_size - 1) // 2
        pad_right = self.kernel_size - 1 - pad_left
        front = x[:, :1, :].repeat(1, pad_left, 1)
        end = x[:, -1:, :].repeat(1, pad_right, 1)
        x_pad = torch.cat([front, x, end], dim=1)
        return self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.moving_average = MovingAverage(kernel_size)

    def forward(self, x: torch.Tensor):
        trend = self.moving_average(x)
        seasonal = x - trend
        return seasonal, trend


class DLinearSeriesCalibration(nn.Module):
    def __init__(self, seq_len: int, pred_len: int, n_series: int, moving_avg_kernel: int = 25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.decomposition = SeriesDecomposition(moving_avg_kernel)
        self.linear_seasonal = nn.Linear(self.seq_len, self.pred_len)
        self.linear_trend = nn.Linear(self.seq_len, self.pred_len)
        self.series_bias = nn.Embedding(n_series, self.pred_len)
        self._init_weights()

    def _init_weights(self):
        nn.init.constant_(self.linear_seasonal.weight, 1.0 / self.seq_len)
        nn.init.constant_(self.linear_trend.weight, 1.0 / self.seq_len)
        nn.init.zeros_(self.linear_seasonal.bias)
        nn.init.zeros_(self.linear_trend.bias)
        nn.init.zeros_(self.series_bias.weight)

    def forward(self, x: torch.Tensor, series_idx: torch.Tensor) -> torch.Tensor:
        seasonal, trend = self.decomposition(x)
        seasonal = seasonal.permute(0, 2, 1)
        trend = trend.permute(0, 2, 1)
        out = self.linear_seasonal(seasonal) + self.linear_trend(trend)
        out = out.permute(0, 2, 1).squeeze(-1)
        return out + self.series_bias(series_idx)


def weighted_mae_loss(pred: torch.Tensor, target: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    return (torch.abs(pred - target) * weights).sum() / weights.sum().clamp_min(1.0)

## Training helpers

In [ ]:
def make_loaders(input_weeks: int, batch_size: int):
    train_ds = WindowDataset(values, holiday_flags, input_weeks, CONFIG["validation_weeks"], split_pos)
    val_ds = ValidationDataset(values, holiday_flags, input_weeks, CONFIG["validation_weeks"], split_pos)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    return train_ds, val_ds, train_loader, val_loader


def evaluate_model(model: nn.Module, loader: DataLoader, device: str):
    model.eval()
    pred_batches, target_batches, weight_batches = [], [], []
    norm_loss_total = 0.0
    norm_weight_total = 0.0
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            weights = batch["weights"].to(device)
            series_idx = batch["series_idx"].to(device)
            pred_norm = model(x, series_idx)
            norm_loss_total += float((torch.abs(pred_norm - y) * weights).sum().cpu())
            norm_weight_total += float(weights.sum().cpu())

            mean = batch["mean"].to(device).unsqueeze(1)
            std = batch["std"].to(device).unsqueeze(1)
            pred = (pred_norm * std + mean).clamp_min(0.0)
            target = y * std + mean
            pred_batches.append(pred.cpu().numpy())
            target_batches.append(target.cpu().numpy())
            weight_batches.append(batch["weights"].cpu().numpy())

    preds = np.concatenate(pred_batches, axis=0)
    targets = np.concatenate(target_batches, axis=0)
    weights = np.concatenate(weight_batches, axis=0)
    return {
        "preds": preds,
        "targets": targets,
        "weights": weights,
        "normalized_wmae": norm_loss_total / max(norm_weight_total, 1.0),
        "wmae": float(np.sum(np.abs(preds - targets) * weights) / np.sum(weights)),
    }


def train_once(params: dict, trial_number: int | None = None, log_wandb: bool = True):
    device = CONFIG["device"]
    train_ds, val_ds, train_loader, val_loader = make_loaders(params["input_weeks"], params["batch_size"])
    model = DLinearSeriesCalibration(
        seq_len=params["input_weeks"],
        pred_len=CONFIG["validation_weeks"],
        n_series=len(sales_panel),
        moving_avg_kernel=params["moving_avg_kernel"],
    ).to(device)

    main_params, series_params = [], []
    for name, param in model.named_parameters():
        if "series_bias" in name:
            series_params.append(param)
        else:
            main_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": main_params, "weight_decay": params["weight_decay"]},
        {"params": series_params, "weight_decay": params["series_bias_weight_decay"]},
    ], lr=params["learning_rate"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=CONFIG["scheduler_patience"])

    run = None
    if log_wandb:
        run = wandb.init(
            project=CONFIG["wandb_project"],
            entity=CONFIG["wandb_entity"],
            group=CONFIG["wandb_group"],
            job_type="hparam_trial" if trial_number is not None else "best_refit",
            name=f"dlinear_tuning_trial_{trial_number:03d}" if trial_number is not None else "dlinear_tuned_best",
            config={**CONFIG, **params, "trial_number": trial_number, "train_windows": len(train_ds)},
            reinit=True,
        )

    best_wmae = math.inf
    best_epoch = -1
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, CONFIG["max_epochs"] + 1):
        model.train()
        train_loss_total = 0.0
        train_weight_total = 0.0
        for batch in train_loader:
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            weights = batch["weights"].to(device)
            series_idx = batch["series_idx"].to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x, series_idx)
            loss = weighted_mae_loss(pred, y, weights)
            loss.backward()
            if CONFIG["clip_grad_norm"]:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
            optimizer.step()
            train_loss_total += float((torch.abs(pred.detach() - y) * weights).sum().cpu())
            train_weight_total += float(weights.sum().cpu())

        train_loss = train_loss_total / max(train_weight_total, 1.0)
        val_result = evaluate_model(model, val_loader, device)
        scheduler.step(val_result["wmae"])
        metrics = {
            "epoch": epoch,
            "train/normalized_wmae_loss": train_loss,
            "validation/normalized_wmae_loss": val_result["normalized_wmae"],
            "validation/wmae": val_result["wmae"],
            "validation/improvement_vs_manual_v1_pct": 100.0 * (CONFIG["best_manual_v1_wmae"] - val_result["wmae"]) / CONFIG["best_manual_v1_wmae"],
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        if run is not None:
            wandb.log(metrics)

        if val_result["wmae"] < best_wmae:
            best_wmae = val_result["wmae"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= CONFIG["patience"]:
                break

    model.load_state_dict(best_state)
    best_val_result = evaluate_model(model, val_loader, device)
    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_validation_wmae"] = best_val_result["wmae"]
        wandb.summary["best_improvement_vs_manual_v1_pct"] = 100.0 * (CONFIG["best_manual_v1_wmae"] - best_val_result["wmae"]) / CONFIG["best_manual_v1_wmae"]
        run.finish()

    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_validation_wmae": best_val_result["wmae"],
        "best_val_result": best_val_result,
        "params": params,
    }

## Run Optuna tuning

In [ ]:
def suggest_params(trial: optuna.Trial) -> dict:
    input_weeks = trial.suggest_categorical("input_weeks", [39, 52])
    # 65 is mathematically possible but gives one window per series, so we exclude it from tuning.
    return {
        "input_weeks": input_weeks,
        "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
        "learning_rate": trial.suggest_float("learning_rate", 2e-4, 1.2e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "series_bias_weight_decay": trial.suggest_float("series_bias_weight_decay", 1e-5, 1e-2, log=True),
        "moving_avg_kernel": trial.suggest_categorical("moving_avg_kernel", [13, 25, 39]),
    }


def objective(trial: optuna.Trial) -> float:
    params = suggest_params(trial)
    result = train_once(params, trial_number=trial.number, log_wandb=True)
    trial.set_user_attr("best_epoch", result["best_epoch"])
    return result["best_validation_wmae"]

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", study_name="dlinear_v1_hparam_tuning", sampler=sampler)
study.optimize(objective, n_trials=CONFIG["n_trials"], show_progress_bar=True)

print("Best trial:", study.best_trial.number)
print("Best WMAE:", study.best_value)
print("Best params:", study.best_params)

study_df = study.trials_dataframe()
study_path = OUTPUT_DIR / "dlinear_hparam_trials.csv"
study_df.to_csv(study_path, index=False)
display(study_df.sort_values("value").head(10))

## Train/log best tuned configuration

In [ ]:
best_params = dict(study.best_params)
if CONFIG["run_final_best"]:
    best_result = train_once(best_params, trial_number=None, log_wandb=True)
    best_model = best_result["model"]
    best_val_result = best_result["best_val_result"]
    best_epoch = best_result["best_epoch"]
else:
    best_result = None
    best_model = None
    best_val_result = None
    best_epoch = None

summary = {
    "experiment": "dlinear_hyperparameter_tuning_v1_architecture",
    "best_trial": int(study.best_trial.number),
    "best_validation_wmae": float(study.best_value),
    "best_params": best_params,
    "manual_v1_wmae": float(CONFIG["best_manual_v1_wmae"]),
    "improvement_vs_manual_v1_pct": float(100.0 * (CONFIG["best_manual_v1_wmae"] - study.best_value) / CONFIG["best_manual_v1_wmae"]),
}
summary_path = OUTPUT_DIR / "dlinear_hparam_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
summary

## Save best model artifact if final best was trained

In [ ]:
if best_model is not None:
    checkpoint_path = OUTPUT_DIR / "dlinear_tuned_best_checkpoint.pt"
    torch.save({
        "model_state_dict": best_model.state_dict(),
        "config": CONFIG,
        "best_params": best_params,
        "best_epoch": best_epoch,
        "best_validation_wmae": best_val_result["wmae"],
        "series_index": sales_panel.index.tolist(),
        "train_dates": [str(pd.Timestamp(d).date()) for d in all_train_dates],
        "validation_dates": [str(pd.Timestamp(d).date()) for d in val_dates],
    }, checkpoint_path)

    preds = best_val_result["preds"]
    targets = best_val_result["targets"]
    records = []
    for row_idx, (store, dept) in enumerate(sales_panel.index):
        for horizon_idx, date in enumerate(val_dates):
            records.append({
                "Store": int(store),
                "Dept": int(dept),
                "Date": pd.Timestamp(date),
                "IsHoliday": bool(holiday_by_date.loc[date]),
                "Weekly_Sales": float(targets[row_idx, horizon_idx]),
                "Prediction": float(preds[row_idx, horizon_idx]),
                "AbsError": float(abs(targets[row_idx, horizon_idx] - preds[row_idx, horizon_idx])),
            })
    val_pred_df = pd.DataFrame(records)
    val_pred_path = OUTPUT_DIR / "dlinear_tuned_best_validation_predictions.csv"
    val_pred_df.to_csv(val_pred_path, index=False)

    fig, ax = plt.subplots(figsize=(7, 4))
    sample = val_pred_df.sample(min(5000, len(val_pred_df)), random_state=SEED)
    ax.scatter(sample["Weekly_Sales"], sample["Prediction"], s=8, alpha=0.25)
    max_axis = np.nanpercentile(sample[["Weekly_Sales", "Prediction"]].to_numpy(), 99)
    ax.plot([0, max_axis], [0, max_axis], color="red", linewidth=1)
    ax.set_title("DLinear tuned best validation predictions")
    ax.set_xlabel("Actual Weekly_Sales")
    ax.set_ylabel("Prediction")
    plt.tight_layout()
    plot_path = OUTPUT_DIR / "dlinear_tuned_best_validation_scatter.png"
    fig.savefig(plot_path, dpi=160)
    plt.show()

    artifact = wandb.Artifact("dlinear-tuned-best-series-calibration", type="model")
    artifact.add_file(str(checkpoint_path))
    artifact.add_file(str(summary_path))
    artifact.add_file(str(study_path))
    artifact.add_file(str(val_pred_path))
    artifact.add_file(str(plot_path))

    artifact_run = wandb.init(
        project=CONFIG["wandb_project"],
        entity=CONFIG["wandb_entity"],
        group=CONFIG["wandb_group"],
        job_type="artifact_logging",
        name="dlinear_tuned_best_artifact",
        config={**CONFIG, **best_params, **summary},
        reinit=True,
    )
    artifact_run.log_artifact(artifact, aliases=["tuned-best", "latest"])
    wandb.log({
        "validation/prediction_table": wandb.Table(dataframe=val_pred_df.sample(min(20000, len(val_pred_df)), random_state=SEED)),
        "validation/scatter": wandb.Image(str(plot_path)),
    })
    artifact_run.finish()
else:
    print("Final best training skipped; no model artifact logged.")